# Sprint 4: Advanced Predictive Modeling (Phase B)
## XGBoost Churn Prediction & Autoencoder Anomaly Detection

This notebook trains our supervised classification model (XGBoost) to predict churn probability, tuning hyperparameters, and visualizing results using SHAP.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

# Set project root in path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src import config
from src.clustering.kmeans_baseline import preprocess_features
from src.models.label_generation import generate_churn_labels
from src.models.churn_prediction import train_xgboost_model

plt.style.use('seaborn-v0_8-whitegrid')

### 1. Load Data & Generate Target Label

In [ ]:
data_path = os.path.join(config.PROCESSED_DATA_DIR, "customer_feature_matrix.csv")

if not os.path.exists(data_path):
    print("Using Dummy Data for Notebook Demo")
    df = pd.DataFrame({
        'customer_id': range(1000),
        'recency_days': np.random.randint(1, 100, 1000),
        'frequency_total': np.random.randint(1, 50, 1000),
        'monetary_value_total': np.random.lognormal(mean=5, sigma=1, size=1000),
        'avg_order_value': np.random.lognormal(mean=3, sigma=0.5, size=1000),
        'spend_last_30d': np.random.randint(0, 500, 1000)
    })
else:
    df = pd.read_csv(data_path)

# Apply 60-day threshold
df = generate_churn_labels(df, churn_threshold_days=60)

print(f"Target distribution:\n{df['is_churned'].value_counts(normalize=True)}")

### 2. Train XGBoost Pipeline

In [ ]:
# Re-run preprocessor to ensure it is trained if testing fresh
_, _ = preprocess_features(df, is_training=True)

model, metrics, feature_names = train_xgboost_model(df)
print("\n--- Model Validation ---")
for k, v in metrics.items():
    print(f"{k.upper()}: {v:.4f}")

### 3. Model Explainability with SHAP

In [ ]:
# Preprocess the entire dataset for SHAP
X_scaled, _ = preprocess_features(df, is_training=False)

# Select a background sample for fast SHAP computation
background_idx = np.random.choice(X_scaled.shape[0], 100, replace=False)
X_background = X_scaled[background_idx]

# Initialize Explainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_scaled[:500]) # Compute for first 500 rows for demo

plot_dir = os.path.join(config.BASE_DIR, 'plots', 'modeling')
os.makedirs(plot_dir, exist_ok=True)

# 1. Summary Beeswarm Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_scaled[:500], feature_names=feature_names, show=False)
plt.title("SHAP Feature Importance & Impact (Beeswarm)")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'shap_summary_plot.png'), dpi=300)
plt.show()